# TADS23 · Lecture 8 · From LOIN to SHACL
Check the worked example against its level of information need (ISO 7817-1), written as SHACL shapes, with **pySHACL** in Google Colab.

## Before you start
**Where:** this notebook runs in **Google Colab**, in your browser. Nothing is installed on your computer; you need a Google account.

1. Open the notebook from the CPO repository in Colab: [open in Colab](https://colab.research.google.com/github/RichZele/cpo/blob/main/notebooks/cpo-lecture8-shacl.ipynb). Then **File › Save a copy in Drive**, and work in your copy.
2. Run the cells **one at a time**, when I say the number: click the ▶ button to the left of the cell, or press **Shift + Enter**.
3. The first run takes about 20 seconds while Colab starts.

**Packages:** Run 1 installs **pySHACL**; it brings **rdflib** with it. **pandas** is already in Colab. Nothing else to install.

**Files:**
- **In class (Runs 1 to 6): no files to upload.** Run 2 reads the school data from the CPO repository on GitHub; this lecture's data and the shapes are written in the cells.
- **For E5 (Run 7):** upload two Turtle files of your own project: your **data** (your graph, with your classification) and your **shapes** (your LOIN table as SHACL).

**If something goes wrong:** Runtime › Restart session, then run the cells again from Run 1.

## ► Run 1 · Install pySHACL
⏸ *Wait, and run it when I say **Run 1**.*

In [ ]:
# ► Run 1 · Install pySHACL (it brings rdflib), then import
!pip install -q pyshacl
import rdflib, pyshacl, urllib.request
print('rdflib', rdflib.__version__, '· pyshacl', pyshacl.__version__)

## ► Run 2 · The data graph
The school data from the CPO repository, plus this lecture's data: the classification links and two Duplex rooms used as classrooms.

⏸ *Wait, and run it when I say **Run 2**.*

In [ ]:
# ► Run 2 · The data graph
import rdflib, urllib.request
SCHOOL = 'https://raw.githubusercontent.com/RichZele/cpo/main/data/cpo-school-data.ttl'
LECTURE7 = '''@prefix cpo:     <https://w3id.org/cpo#> .
@prefix inst:    <https://w3id.org/cpo/data/> .
@prefix ifcdata: <https://w3id.org/cpo/duplex/> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix rdfs:    <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:     <http://www.w3.org/2001/XMLSchema#> .
@prefix props:   <https://w3id.org/props#> .
@prefix uc:      <https://uniclass.thenbs.com/taxon/> .
@prefix iec:     <https://w3id.org/cpo/class/iec81346-2/> .

# =====================================================================
#  CPO worked example, Lecture 7: classification links and two Duplex rooms used as classrooms
#  Classes are linked with dcterms:subject (Dublin Core, used with SKOS for "is about / is classified as").
# =====================================================================

# ---- the school: spaces and luminaire types classified
inst:classroom_2_14 dcterms:subject uc:sl_25_10_14 .
inst:classroom_2_15 dcterms:subject uc:sl_25_10_14 .
inst:type_23784-402 dcterms:subject uc:pr_70_70_48_45 , iec:UAC .
inst:type_23916-529 dcterms:subject uc:pr_70_70_48_45 , iec:UAC .
inst:type_73029-520 dcterms:subject uc:pr_70_70_48_70 , iec:UAC .

# ---- the Duplex: two rooms from the IFC model, now used as classrooms
#      Their area comes from IFC as props:area (a plain number, no unit); the LOIN asks for cpo:hasArea.
ifcdata:space_0BTBFw6f90Nfh9rP1dlXr2 a cpo:Classroom ;
    rdfs:label "A102" ;
    props:area 30.14165 ;
    cpo:hasLightingRequirement inst:req_EN12464_classroom ;
    dcterms:subject uc:sl_25_10_14 .

ifcdata:space_0BTBFw6f90Nfh9rP1dl_CZ a cpo:Classroom ;
    rdfs:label "B102" ;
    props:area 30.14165 ;
    dcterms:subject uc:sl_25_10_14 .
'''
CLASSES = '''@prefix skos:    <http://www.w3.org/2004/02/skos/core#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix rdfs:    <http://www.w3.org/2000/01/rdf-schema#> .
@prefix uc:      <https://uniclass.thenbs.com/taxon/> .
@prefix iec:     <https://w3id.org/cpo/class/iec81346-2/> .

# =====================================================================
#  CPO worked example, Lecture 7: classification as SKOS concept schemes
#  Uniclass 2015 follows ISO 12006-2; its codes are published at uniclass.thenbs.com,
#  so the taxon pages are used as IRIs (Spaces/locations v1.37, October 2026).
#  IEC 81346-2:2019, Table 1 and Table 2: classes of objects by inherent function.
#  Only the classes the worked example needs.
# =====================================================================

# ---- Uniclass 2015, table SL: spaces and locations
uc:sl a skos:ConceptScheme ;
    skos:prefLabel "Uniclass 2015, Spaces/ locations"@en ;
    dcterms:source "ISO 12006-2 framework, Uniclass 2015 (NBS)" .

uc:sl_25 a skos:Concept ; skos:inScheme uc:sl ; skos:topConceptOf uc:sl ;
    skos:notation "SL_25" ; skos:prefLabel "Cultural, educational, scientific and information spaces"@en .
uc:sl_25_10 a skos:Concept ; skos:inScheme uc:sl ; skos:broader uc:sl_25 ;
    skos:notation "SL_25_10" ; skos:prefLabel "Educational spaces"@en .
uc:sl_25_10_14 a skos:Concept ; skos:inScheme uc:sl ; skos:broader uc:sl_25_10 ;
    skos:notation "SL_25_10_14" ; skos:prefLabel "Classrooms"@en .

# ---- Uniclass 2015, table Pr: products
uc:pr a skos:ConceptScheme ;
    skos:prefLabel "Uniclass 2015, Products"@en ;
    dcterms:source "ISO 12006-2 framework, Uniclass 2015 (NBS)" .

uc:pr_70 a skos:Concept ; skos:inScheme uc:pr ; skos:topConceptOf uc:pr ;
    skos:notation "Pr_70" ; skos:prefLabel "Services and process outlet products"@en .
uc:pr_70_70 a skos:Concept ; skos:inScheme uc:pr ; skos:broader uc:pr_70 ;
    skos:notation "Pr_70_70" ; skos:prefLabel "Power and lighting outlet products"@en .
uc:pr_70_70_48 a skos:Concept ; skos:inScheme uc:pr ; skos:broader uc:pr_70_70 ;
    skos:notation "Pr_70_70_48" ; skos:prefLabel "Luminaires, lighting track and lighting accessories"@en .
uc:pr_70_70_48_45 a skos:Concept ; skos:inScheme uc:pr ; skos:broader uc:pr_70_70_48 ;
    skos:notation "Pr_70_70_48_45" ; skos:prefLabel "LED light panels"@en .
uc:pr_70_70_48_70 a skos:Concept ; skos:inScheme uc:pr ; skos:broader uc:pr_70_70_48 ;
    skos:notation "Pr_70_70_48_70" ; skos:prefLabel "Recessed downlights"@en .

# ---- IEC 81346-2:2019: classes of objects by inherent function (product aspect)
iec:scheme a skos:ConceptScheme ;
    skos:prefLabel "IEC 81346-2:2019, classes of objects"@en .

iec:E a skos:Concept ; skos:inScheme iec:scheme ; skos:topConceptOf iec:scheme ;
    skos:notation "E" ; skos:prefLabel "emitting object"@en ; skos:definition "object for emitting"@en .
iec:EA a skos:Concept ; skos:inScheme iec:scheme ; skos:broader iec:E ;
    skos:notation "EA" ; skos:prefLabel "light object"@en ; skos:definition "emitting object of light"@en .
iec:EAA a skos:Concept ; skos:inScheme iec:scheme ; skos:broader iec:EA ;
    skos:notation "EAA" ; skos:prefLabel "electric lamp"@en ; skos:definition "light object by electricity"@en .
iec:U a skos:Concept ; skos:inScheme iec:scheme ; skos:topConceptOf iec:scheme ;
    skos:notation "U" ; skos:prefLabel "holding object"@en ; skos:definition "object for localising of other objects"@en .
iec:UA a skos:Concept ; skos:inScheme iec:scheme ; skos:broader iec:U ;
    skos:notation "UA" ; skos:prefLabel "positioning object"@en ; skos:definition "holding object by fixing at a defined position"@en .
iec:UAC a skos:Concept ; skos:inScheme iec:scheme ; skos:broader iec:UA ;
    skos:notation "UAC" ; skos:prefLabel "light fixture"@en ; skos:definition "positioning object for light source"@en .
'''
data = rdflib.Graph()
data.parse(data=urllib.request.urlopen(SCHOOL).read().decode('utf-8'), format='turtle')
data.parse(data=LECTURE7, format='turtle')
data.parse(data=CLASSES, format='turtle')
print(len(data), 'triples in the data graph')

## ► Run 3 · The shapes: the LOIN as SHACL
One node shape per object, one property shape per row of the LOIN table.

⏸ *Wait, and run it when I say **Run 3**.*

In [ ]:
# ► Run 3 · The shapes: the LOIN as SHACL
SHAPES = '''@prefix sh:      <http://www.w3.org/ns/shacl#> .
@prefix cpo:     <https://w3id.org/cpo#> .
@prefix qudt:    <http://qudt.org/schema/qudt/> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix skos:    <http://www.w3.org/2004/02/skos/core#> .
@prefix xsd:     <http://www.w3.org/2001/XMLSchema#> .
@prefix unit:    <http://qudt.org/vocab/unit/> .
@prefix shapes:  <https://w3id.org/cpo/shapes/> .

# =====================================================================
#  CPO worked example, Lecture 7: the LOIN for classroom lighting, as SHACL shapes
#  Purpose: check that classroom lighting can be designed and verified (EN 12464-1:2021)
#  Property definitions: CEN/TS 17623:2021, 04-0005 rated luminous flux (lm), 02-0002 rated input power (W)
# =====================================================================

shapes:ClassroomShape a sh:NodeShape ;
    sh:targetClass cpo:Classroom ;
    sh:property [
        sh:path cpo:hasArea ;
        sh:minCount 1 ; sh:maxCount 1 ;
        sh:class qudt:QuantityValue ;
        sh:message "LOIN: a classroom has one floor area, in m2."@en ] ;
    sh:property [
        sh:path cpo:hasLightingRequirement ;
        sh:minCount 1 ;
        sh:message "LOIN: a classroom has a lighting requirement."@en ] ;
    sh:property [
        sh:path dcterms:subject ;
        sh:minCount 1 ;
        sh:class skos:Concept ;
        sh:message "LOIN: a classroom is classified."@en ] .

shapes:LuminaireTypeShape a sh:NodeShape ;
    sh:targetClass cpo:LuminaireType ;
    sh:property [
        sh:path ( cpo:hasLuminousFlux qudt:numericValue ) ;
        sh:minCount 1 ;
        sh:datatype xsd:decimal ;
        sh:minExclusive 0 ;
        sh:message "LOIN: rated luminous flux, in lm, greater than 0."@en ] ;
    sh:property [
        sh:path ( cpo:hasLuminousFlux qudt:unit ) ;
        sh:hasValue unit:LM ;
        sh:maxCount 1 ;
        sh:message "LOIN: rated luminous flux in lumen, unit:LM."@en ] ;
    sh:property [
        sh:path ( cpo:hasRatedPower qudt:numericValue ) ;
        sh:minCount 1 ;
        sh:datatype xsd:decimal ;
        sh:minExclusive 0 ;
        sh:message "LOIN: rated input power, in W, greater than 0."@en ] ;
    sh:property [
        sh:path ( cpo:hasRatedPower qudt:unit ) ;
        sh:hasValue unit:W ;
        sh:maxCount 1 ;
        sh:message "LOIN: rated input power in watt, unit:W."@en ] ;
    sh:property [
        sh:path dcterms:subject ;
        sh:minCount 1 ;
        sh:message "LOIN: a luminaire type is classified."@en ] .
'''
shapes = rdflib.Graph().parse(data=SHAPES, format='turtle')
print(len(shapes), 'triples in the shapes graph')

## ► Run 4 · Validate
⏸ *Wait, and run it when I say **Run 4**.*

In [ ]:
# ► Run 4 · Validate
import pyshacl
conforms, report_graph, report_text = pyshacl.validate(data, shacl_graph=shapes, inference='none')
print('Conforms:', conforms)
print(report_text)

## ► Run 5 · The violations as a table
⏸ *Wait, and run it when I say **Run 5**.*

In [ ]:
# ► Run 5 · The violations as a table
import pandas as pd
q = '''PREFIX sh: <http://www.w3.org/ns/shacl#>
SELECT ?focus ?path ?message WHERE {
  ?r a sh:ValidationResult ; sh:focusNode ?focus ; sh:resultPath ?path ; sh:resultMessage ?message .
}'''
rows = []
for r in report_graph.query(q):
    label = data.value(r.focus, rdflib.RDFS.label) or r.focus
    rows.append((str(label), data.namespace_manager.normalizeUri(r.path), str(r.message)))
pd.DataFrame(rows, columns=['focus node', 'path', 'message'])

## ► Run 6 · Fix at the source, and validate again
The area of A102 and B102 is in the IFC data as `props:area`, a plain number. The LOIN asks for `cpo:hasArea`, a quantity with a unit. B102 also needs its lighting requirement.

⏸ *Wait, and run it when I say **Run 6**.*

In [ ]:
# ► Run 6 · Fix at the source, and validate again
fix = '''PREFIX cpo:   <https://w3id.org/cpo#>
PREFIX inst:  <https://w3id.org/cpo/data/>
PREFIX props: <https://w3id.org/props#>
PREFIX qudt:  <http://qudt.org/schema/qudt/>
PREFIX unit:  <http://qudt.org/vocab/unit/>
PREFIX xsd:   <http://www.w3.org/2001/XMLSchema#>
INSERT { ?room cpo:hasArea ?q . ?q a qudt:QuantityValue ; qudt:numericValue ?v ; qudt:unit unit:M2 . }
WHERE  { ?room a cpo:Classroom ; props:area ?a .
         FILTER NOT EXISTS { ?room cpo:hasArea ?any }
         BIND (xsd:decimal(ROUND(?a * 100) / 100) AS ?v)
         BIND (IRI(CONCAT(STR(?room), '_area')) AS ?q) } ;
INSERT { ?room cpo:hasLightingRequirement inst:req_EN12464_classroom . }
WHERE  { ?room a cpo:Classroom . FILTER NOT EXISTS { ?room cpo:hasLightingRequirement ?r } }'''
data.update(fix)
conforms, report_graph, report_text = pyshacl.validate(data, shacl_graph=shapes, inference='none')
print('Conforms:', conforms)

## ► Run 7 · Your turn, E5: validate your own project
Upload **two Turtle files** from your project:
1. your **data**: your graph from E3, with your classification from E4 (one file, or merge them first);
2. your **shapes**: your LOIN table written as SHACL.

Run the cell, click **Choose Files**, and select **both files at once** (hold Ctrl). The cell waits until you have chosen them. The file whose name contains *shape* is used as the shapes graph.

In [ ]:
# ► Run 7 · Your turn, E5: validate your own project
import rdflib, pyshacl, pandas as pd
from google.colab import files
up = files.upload()                      # choose your data file and your shapes file
shape_names = [n for n in up if 'shape' in n.lower()]
data_names = [n for n in up if n not in shape_names]
if not shape_names or not data_names:
    print('Upload two files: one data file, and one shapes file with "shape" in its name.')
else:
    my_data, my_shapes = rdflib.Graph(), rdflib.Graph()
    for n in data_names:
        my_data.parse(data=up[n].decode('utf-8'), format='turtle')
    for n in shape_names:
        my_shapes.parse(data=up[n].decode('utf-8'), format='turtle')
    print(len(my_data), 'data triples ·', len(my_shapes), 'shapes triples')
    ok, rep, txt = pyshacl.validate(my_data, shacl_graph=my_shapes, inference='none')
    print('Conforms:', ok)
    q = '''PREFIX sh: <http://www.w3.org/ns/shacl#>
    SELECT ?focus ?path ?message WHERE { ?r a sh:ValidationResult ; sh:focusNode ?focus .
      OPTIONAL { ?r sh:resultPath ?path } OPTIONAL { ?r sh:resultMessage ?message } }'''
    rows = [(str(r.focus), str(r.path or ''), str(r.message or '')) for r in rep.query(q)]
    display(pd.DataFrame(rows, columns=['focus node', 'path', 'message']))

Put the validation result in your report, **Results, step 3**: your LOIN-to-SHACL table, and what you did with each violation.